<a href="https://colab.research.google.com/github/Debotri-Ghosh/G-demo/blob/main/DDP_Assignment3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Course: Distributed Data processing

Name: Debotri Ghosh

Registration No.: 25MML1008


# Assignment 3: Introduction to Spark Structured Streaming

## a) Theory

### What is Structured Streaming?
Structured Streaming is a scalable and fault-tolerant stream processing engine built on Apache Spark. It allows processing of real-time data streams using the same APIs as batch processing.

### Difference between Batch Processing and Structured Streaming

| Feature | Batch Processing | Structured Streaming |
|--------|----------------|----------------------|
| Data Type | Static data | Continuous data stream |
| Processing | Runs on complete dataset | Runs continuously |
| Latency | High | Low (real-time) |
| Use Case | Historical analysis | Live analytics |

### Key Features
- **Scalability**: Handles large-scale data using distributed systems
- **Exactly-once processing**: Ensures no duplicate data
- **Event-time processing**: Processes based on actual event time
- **Fault tolerance**: Recovers automatically from failures

### Real-Time Use Cases
- Fraud detection
- System monitoring
- Stock market analysis
- IoT data processing
- Recommendation systems


b) Practical / Coding
Write a PySpark Structured Streaming program to:
- Read data from a streaming source (e.g., socket or file directory)
- Perform a simple transformation (such as word count)
- Output the results to the console
- Explain each step in comments



In [ ]:
#Step 1: Install & Import
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split, col
import os

In [ ]:
#Step 2: Create Spark Session
spark = SparkSession.builder \
    .appName("StructuredStreaming_Colab") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("✅ Spark ready!")

✅ Spark ready!


In [ ]:
#Step 3: Create Streaming Input Folder
# Create folder
os.makedirs("stream_input", exist_ok=True)

# Add sample data (acts like streaming input)
with open("stream_input/data1.txt", "w") as f:
    f.write("spark is fast and spark is powerful\n")

with open("stream_input/data2.txt", "w") as f:
    f.write("big data processing with spark\n")

print("✅ Input files created!")

✅ Input files created!


In [ ]:
#Step 4: Read Streaming Data
stream_df = spark.readStream \
    .format("text") \
    .option("path", "stream_input") \
    .load()

print("Is Streaming:", stream_df.isStreaming)
stream_df.printSchema()

Is Streaming: True
root
 |-- value: string (nullable = true)



In [ ]:
#Step 5: Transformations (Word Count)
# Split words
words = stream_df.select(
    explode(split(col("value"), " ")).alias("word")
)

# Count words
word_counts = words.groupBy("word").count()

In [ ]:
#Step 6: Output to Console (Action)
query = word_counts.writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", False) \
    .trigger(once=True) \
    .option("checkpointLocation", "checkpoint") \
    .start()

query.awaitTermination()

Structured Streaming Pipeline

| **Step** | **Operation**           | **Description**                                                                         |
| -------- | ----------------------- | --------------------------------------------------------------------------------------- |
| **1**    | `readStream`            | Continuously monitors the input folder and reads incoming data as a streaming DataFrame |
| **2**    | `split()` + `explode()` | Breaks each line into individual words and converts them into separate rows             |
| **3**    | `groupBy()` + `count()` | Groups identical words and calculates their frequency (word count)                      |
| **4**    | `writeStream`           | Defines the output destination (console, file, etc.) for streaming results              |
| **5**    | `trigger(once=True)`    | Processes all available data once and stops automatically (ideal for Colab/Jupyter)     |

